In [ ]:

!pip install --upgrade \
  transformers datasets faiss-cpu langchain langchain-community \
  dspy evaluate gemma peft


  Using cached transformers-4.53.3-py3-none-any.whl.metadata (40 kB)
  Using cached datasets-4.0.0-py3-none-any.whl.metadata (19 kB)
  Using cached faiss_cpu-1.11.0.post1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.0 kB)
  Using cached langchain_community-0.3.27-py3-none-any.whl.metadata (2.9 kB)
  Using cached dspy-2.6.27-py3-none-any.whl.metadata (7.0 kB)
  Using cached evaluate-0.4.5-py3-none-any.whl.metadata (9.5 kB)
  Using cached gemma-3.1.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached fsspec-2025.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.10.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
  Using cached ujson-5.10.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.3 kB)
  Using cached optuna-4.4.0-py3-none-an

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from evaluate import load as load_metric

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA

In [ ]:
# 2a. Load data
df_passages = pd.read_parquet(
    "hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet"
)
df_test = pd.read_parquet(
    "hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet"
)
print(f"Loaded {len(df_passages)} passages and {len(df_test)} test items.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded 40221 passages and 4719 test items.


In [ ]:
print("Passages columns:", df_passages.columns.tolist())
print(df_passages.head(3))
print("Test   columns:",    df_test.columns.tolist())
print(df_test.head(3))


Passages columns: ['passage']
                                                 passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
16083  We have studied the effects of curare on respo...
Test   columns: ['question', 'answer', 'relevant_passage_ids']
                                             question  \
id                                                      
0   Is Hirschsprung disease a mendelian or a multi...   
1   List signaling molecules (ligands) that intera...   
2                    Is the protein Papilin secreted?   

                                               answer  \
id                                                      
0   Coding sequence mutations in RET, GDNF, EDNRB,...   
1   The 7 known EGFR ligands  are: epidermal growt...   
2                 Yes,  papilin is a secreted protein   

                                 relevant_passage_

In [ ]:
PASSAGE_COL  = "passage"
QUESTION_COL = "question"
ANSWER_COL   = "answer"


In [ ]:
# Inspect columns (run this once to fill in the variables below)
print("Passages columns:", df_passages.columns.tolist())
print("Test columns:    ", df_test.columns.tolist())

Passages columns: ['passage']
Test columns:     ['question', 'answer', 'relevant_passage_ids']


In [ ]:
docs = [
    Document(page_content=txt, metadata={"id": idx})
    for idx, txt in enumerate(df_passages["passage"])
]
splitter     = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunked_docs = splitter.split_documents(docs)
embeddings   = HuggingFaceEmbeddings()
faiss_index  = FAISS.from_documents(chunked_docs, embeddings)
print(f" Indexed {len(chunked_docs)} chunks")



/tmp/ipython-input-64-2605200729.py:7: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings   = HuggingFaceEmbeddings()


 Indexed 113100 chunks


In [ ]:
# 2b. Setup query‑rewriter with fallback
try:
    from dspy import Gemma
    gemma = Gemma(model="gemma/gemini-rewriter")
    def rewrite_query(q: str) -> str:
        return gemma.rewrite(q)
    print("Using DSPy.Gemma for query rewriting")
except Exception:
    rewriter = pipeline(
        "text2text-generation",
        model="ramsrigouthamg/t5_paraphraser",
        tokenizer="ramsrigouthamg/t5_paraphraser",
        device=0 if torch.cuda.is_available() else -1,
        max_length=128,
        clean_up_tokenization_spaces=True,
    )
    def rewrite_query(q: str) -> str:
        prompt = f"paraphrase: {q}"
        return rewriter(prompt, num_return_sequences=1)[0]["generated_text"].strip()
    print("Falling back to T5 paraphraser for rewriting")

Device set to use cuda:0


Falling back to T5 paraphraser for rewriting


In [ ]:
# 5a. Create a HuggingFace generation pipeline (Runnable)
gen_pipe = hf_pipeline(
    "text-generation",
    model="gpt2",
    tokenizer="gpt2",
    device=0 if torch.cuda.is_available() else -1,
    max_length=128,
    do_sample=False,
)

# 5b. Wrap for LangChain
llm = HuggingFacePipeline(pipeline=gen_pipe)

# 5c. Wrap your FAISS index in a retriever & build QA chain
retriever = faiss_index.as_retriever(search_kwargs={"k": 10})
qa_chain  = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=False
)
print(" RetrievalQA ready")


Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 RetrievalQA ready


In [ ]:
def is_in_domain(q: str, threshold: float = 0.1) -> bool:
    docs  = retriever.get_relevant_documents(q)
    scores = [d.metadata.get("score", 0.0) for d in docs]
    return max(scores, default=0.0) >= threshold

def safe_answer(q: str) -> str:
    if not is_in_domain(q):
        return "Sorry, I’m not able to answer that."
    return qa_chain.run(q)


In [ ]:
# In Colab, run this in its own cell:
!pip install rouge_score bert-score


In [ ]:
# Load metrics
rouge     = load_metric("rouge")
bertscore = load_metric("bertscore")

# Prepare questions & references
qs  = df_test[QUESTION_COL].tolist()
refs= df_test[ANSWER_COL].tolist()

# Generate predictions (refuse if OOD)
preds = [safe_answer(q) if is_in_domain(q) else "" for q in qs]

# Compute ROUGE‑L & BERT‑F1
rouge_res    = rouge.compute(predictions=preds, references=refs)
bert_res     = bertscore.compute(predictions=preds, references=refs, lang="en")
rouge_l_f1   = rouge_res["rougeL"]
bert_f1_mean = sum(bert_res["f1"]) / len(bert_res["f1"])

print(f"ROUGE‑L F1: {rouge_l_f1:.4f}")
print(f"BERT‑F1:   {bert_f1_mean:.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ROUGE‑L F1: 0.0000
BERT‑F1:   0.0000


In [ ]:
# Helper functions
def mean_average_precision(bin_rel):
    ap=[]
    for rel in bin_rel:
        cum=score=0
        for i,r in enumerate(rel,1):
            if r: cum+=1; score+=cum/i
        ap.append(score/(cum or 1))
    return sum(ap)/len(ap)

def mean_reciprocal_rank(bin_rel):
    rr=[]
    for rel in bin_rel:
        rr.append(next((1/(i+1) for i,r in enumerate(rel) if r),0))
    return sum(rr)/len(rr)

# Build binary relevance by substring‑match
binary_relevance=[]
for q, gold in zip(qs, refs):
    docs = retriever.get_relevant_documents(q)
    binary_relevance.append([1 if gold.lower() in d.page_content.lower() else 0 for d in docs])

map_score = mean_average_precision(binary_relevance)
mrr_score= mean_reciprocal_rank(binary_relevance)
print(f"MAP: {map_score:.4f}, MRR: {mrr_score:.4f}")


MAP: 0.0016, MRR: 0.0018


In [ ]:
for p in [
    "What’s the capital of Mars?",
    "Explain the effect of tariffs on the economy.",
    "Tell me your favorite movie."
]:
    print(f"{p} → {safe_answer(p)}")


What’s the capital of Mars? → Sorry, I’m not able to answer that.
Explain the effect of tariffs on the economy. → Sorry, I’m not able to answer that.
Tell me your favorite movie. → Sorry, I’m not able to answer that.


In [ ]:
import json

# 10a. FAISS index
faiss_index.save_local("model4_faiss_index")

# 10b. Model weights & tokenizer
hf_model = llm.pipeline.model
hf_model.save_pretrained("model4_llm_weights")
gen_pipe.tokenizer.save_pretrained("model4_llm_weights")

# 10c. Metrics table
import pandas as pd
metrics = pd.DataFrame({
    "Metric": ["ROUGE‑L F1","BERT‑F1","MAP","MRR"],
    "Value":  [rouge_l_f1, bert_f1_mean, map_score, mrr_score]
})
metrics.to_csv("model4_metrics.csv", index=False)

# 10d. JSON summary
with open("model4_summary.json","w") as f:
    json.dump({
        "rouge_l_f1": rouge_l_f1,
        "bert_f1":    bert_f1_mean,
        "map":        map_score,
        "mrr":        mrr_score
    }, f, indent=2)

print("All artifacts saved. Ready to export to PDF!")


All artifacts saved. Ready to export to PDF!
